<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/ADPs/Practica/Practica_3_%D0%9C%D0%B8%D0%BD%D0%B8_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82_%D1%81_%D0%B8%D0%BD%D1%81%D1%82%D1%80%D1%83%D0%BC%D0%B5%D0%BD%D1%82%D0%B0%D0%BC%D0%B8_%D0%B8_%D0%BC%D1%83%D0%BB%D1%8C%D1%82%D0%B8%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D0%BD%D0%BE%D0%B9_%D0%BF%D1%80%D0%BE%D0%B2%D0%B5%D1%80%D0%BA%D0%BE%D0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



#  Мини-агент с инструментами и мультиагентной проверкой

## Что мы делаем и зачем

В первом проекте модель просто отвечала. Во втором — отвечала, глядя в текст (RAG). Теперь она будет **действовать**: вызывать инструменты (калькулятор, поиск по глоссарию) и **решать, когда это нужно**. А затем мы построим цепочку из трёх ролей: **ответчик → критик → корректор**. Это простейшая мультиагентная система: три отдельных вызова модели с разными системными промптами, которые последовательно обрабатывают один вопрос.

Главная идея агента: вы даёте модели список инструментов. Модель сама выбирает, какой вызвать. Ваш код выполняет выбранную функцию и возвращает результат обратно модели. Это повторяется, пока модель не скажет: «Готово, вот ответ».

---

## Шаг 0. Что уже должно быть готово (не повторяем)

После первых двух проектов у вас уже установлены и работают:
- Python и VS Code;
- Ollama;
- библиотека `requests`.

**Что нового понадобится:** модель, которая **поддерживает tool calling**. `phi3:mini`, которую вы использовали в первом проекте, для этого не подходит. Официальная документация Ollama и каталог моделей на ноябрь 2026 года указывают, что tool calling поддерживают, в частности, `qwen2.5` (3b, 7b), `llama3.1` (8b и больше), `mistral` (7b), `qwen3` и `gemma4`. Для учебного проекта достаточно **`qwen2.5:3b`** — она маленькая (около 1.9 ГБ), быстро работает и хорошо следует инструкциям.

Скачайте её в PowerShell:
```powershell
ollama pull qwen2.5:3b
```

Проверьте, что модель появилась:
```powershell
ollama list
```

> **Если `qwen2.5:3b` плохо справляется с tool calling** (например, отвечает без вызова инструментов), попробуйте `qwen2.5:7b` (около 4.7 ГБ, нужен ноутбук с 16 ГБ ОЗУ) или `llama3.1:8b` (около 4.9 ГБ). Просто замените строку `MODEL = "qwen2.5:3b"` на новое имя в обоих файлах.

---

## Шаг 1. Создайте папку проекта (подробно, как в учебнике)

1. Откройте **Проводник Windows**.
2. Перейдите в папку `Документы` (или в любое удобное место).
3. Создайте новую папку без пробелов и русских букв: `agent_psychology`.
4. Внутри `agent_psychology` создайте папку `code`. В ней будут лежать скрипты.
5. Откройте **VS Code**.
6. В верхнем меню выберите **File → Open Folder…** (Файл → Открыть папку).
7. Найдите папку `agent_psychology` и нажмите «Выбрать папку».
8. Если VS Code спросит, доверяете ли вы авторам файлов, нажмите **Yes, I trust the authors**.

Структура:
```
agent_psychology/
└── code/
```

---

## Шаг 2. Создайте простого агента с одним инструментом

Начнём с минимального примера: модель умеет вызывать **калькулятор** (сложение двух чисел). Это позволит увидеть весь цикл tool calling без лишней сложности.

### 2.1. Создайте файл `simple_agent.py` в папке `code`

В левой панели Explorer наведите курсор на папку `code` → нажмите иконку **New File** → введите `simple_agent.py` → `Enter`.

### 2.2. Вставьте код

```python
import requests

# ---------------------------------------------------------------
# НАСТРОЙКИ
# ---------------------------------------------------------------
CHAT_URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5:3b"          # модель, поддерживающая tool calling


# ---------------------------------------------------------------
# ИНСТРУМЕНТ (tool) — обычная Python-функция
# ---------------------------------------------------------------
def add_numbers(a: float, b: float) -> str:
    """Складывает два числа и возвращает результат строкой."""
    result = a + b
    return f"Сумма {a} и {b} равна {result}"


# Описание инструмента для модели (JSON Schema).
# Именно description помогает модели понять, КОГДА вызывать инструмент.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Складывает два числа. Используй, когда нужно посчитать сумму.",
            "parameters": {
                "type": "object",
                "required": ["a", "b"],
                "properties": {
                    "a": {"type": "number", "description": "Первое число"},
                    "b": {"type": "number", "description": "Второе число"},
                },
            },
        },
    }
]


def call_model(messages, tools=TOOLS):
    """Отправляет сообщения в Ollama и возвращает ответ модели целиком."""
    payload = {
        "model": MODEL,
        "messages": messages,
        "tools": tools,
        "stream": False,
        "options": {"temperature": 0.2},
    }
    response = requests.post(CHAT_URL, json=payload, timeout=300)
    response.raise_for_status()
    return response.json()["message"]


def agent_loop(user_question, max_steps=5):
    """Главный цикл агента: модель решает, что делать, пока не даст ответ."""
    messages = [{"role": "user", "content": user_question}]

    for step in range(1, max_steps + 1):
        print(f"\n--- Шаг {step} ---")
        message = call_model(messages)

        # Если модель НЕ просит инструмент — она даёт финальный ответ
        if not message.get("tool_calls"):
            print("Модель дала финальный ответ.")
            # or "" защищает от случая, когда content пришёл как None
            return message.get("content") or ""

        # Сообщение ассистента с tool_calls добавляем в историю ОДИН раз —
        # до цикла по вызовам. Это важно: если модель вернёт два вызова
        # за один шаг, сообщение не должно дублироваться.
        print("Модель решила вызвать инструмент(ы):")
        messages.append(message)

        # Обрабатываем каждый вызов инструмента
        for tool_call in message["tool_calls"]:
            name = tool_call["function"]["name"]
            args = tool_call["function"]["arguments"]
            print(f"  {name} с аргументами {args}")

            if name == "add_numbers":
                result = add_numbers(**args)
            else:
                result = f"Неизвестный инструмент: {name}"

            # Добавляем результат работы инструмента
            messages.append(
                {
                    "role": "tool",
                    "tool_name": name,
                    "content": str(result),
                }
            )

    return "Агент не смог завершить работу за отведённое число шагов."


if __name__ == "__main__":
    print("Простой агент-калькулятор. Введите вопрос или 'выход'.")
    while True:
        question = input("\nВы: ").strip()
        if question.lower() in ("выход", "exit", "quit"):
            break
        if not question:
            continue

        answer = agent_loop(question)
        print("\nОтвет агента:")
        print(answer)
```

### 2.3. Что здесь важно понять (актуальные детали на ноябрь 2026)

- **Формат `tools`** — это JSON Schema, совместимая с OpenAI-подобным форматом. Официальная документация Ollama показывает именно такой вид: `{"type": "function", "function": {"name": ..., "description": ..., "parameters": ...}}`.
- **Ответ модели с `tool_calls`** приходит в поле `message.tool_calls`. Каждый вызов содержит `function.name` и `function.arguments`. В актуальной документации аргументы приходят как объект (словарь), и это важно: в коде мы используем их напрямую через `**args`.
- **Обратный вызов** оформляется как сообщение с ролью `"tool"` и полем `"tool_name"`. Именно так показано в официальном примере: `{"role": "tool", "tool_name": "get_temperature", "content": "22°C"}`.
- **Цикл** повторяется, пока в ответе есть `tool_calls`. Ограничение `max_steps=5` защищает от зацикливания (модель иногда «застревает» и пытается вызвать инструмент снова и снова).
- **`messages.append(message)` стоит до цикла `for tool_call`.** Это не случайно: если модель вернёт сразу два вызова инструмента, сообщение ассистента должно появиться в истории ровно один раз, а после него — результаты обоих вызовов. Если добавлять его внутри цикла, история «сломается», и следующий запрос к модели завершится ошибкой или зависанием.
- **`temperature: 0.2`** — низкая температура. Для агента важна предсказуемость: мы хотим, чтобы модель стабильно выбирала правильный инструмент, а не «творила».

### 2.4. Запустите и проверьте

В терминале VS Code (убедитесь, что вы в папке `code`; если нет — `cd code`):
```powershell
python simple_agent.py
```

Задайте вопрос: **«Сколько будет 127 плюс 45?»**

**Что должно произойти:**
1. Модель увидит инструмент `add_numbers`.
2. На первом шаге она вернёт `tool_calls` с аргументами `{"a": 127, "b": 45}`.
3. Ваш код вызовет `add_numbers` и добавит результат `«Сумма 127 и 45 равна 172»` в сообщения с ролью `tool`.
4. На втором шаге модель прочитает результат и даст финальный ответ: «172».

**Если модель сразу отвечает «172» без вызова инструмента** — это не ошибка, но tool calling не сработал. Убедитесь, что используете `qwen2.5:3b` или другую модель из списка поддерживающих, и что вопрос действительно требует вычисления. Некоторые модели могут «угадать» ответ, если он простой. Чтобы надёжнее проверить tool calling, задайте вопрос с большими числами: **«Сколько будет 123456 плюс 789012?»** — такое «в уме» модель посчитает вряд ли, и ей придётся вызвать калькулятор.

---

## Шаг 3. Добавьте второй инструмент — поиск по глоссарию

Теперь сделаем агента полезнее для психологии: добавим инструмент, который ищет определение термина в вашем глоссарии.

### 3.1. Создайте файл `glossary.json` в папке `code`

В папке `code` создайте новый файл `glossary.json` (так же, как создавали `simple_agent.py`) и вставьте:

```json
{
  "катастрофизация": "Когнитивное искажение: представление о том, что произойдёт самое худшее из возможного, без учёта других исходов.",
  "сверхобобщение": "Когнитивное искажение: вывод о том, что одно негативное событие будет повторяться всегда.",
  "персонализация": "Когнитивное искажение: принятие на себя ответственности за события, которые от вас не зависят.",
  "чтение мыслей": "Когнитивное искажение: уверенность в том, что вы знаете, о чём думают другие, без достаточных оснований."
}
```

### 3.2. Обновите `simple_agent.py` — полный код с двумя инструментами

Сейчас мы **полностью заменим** содержимое `simple_agent.py`. Старый код с одним инструментом больше не нужен: новый файл умеет и считать, и искать в глоссарии.

Откройте `simple_agent.py` в редакторе, выделите всё (`Ctrl+A`), удалите и вставьте следующий полный код:

```python
import json
import os
import requests

# ---------------------------------------------------------------
# НАСТРОЙКИ
# ---------------------------------------------------------------
CHAT_URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5:3b"          # модель, поддерживающая tool calling
GLOSSARY_PATH = "glossary.json"


# ---------------------------------------------------------------
# ИНСТРУМЕНТ 1: калькулятор
# ---------------------------------------------------------------
def add_numbers(a: float, b: float) -> str:
    """Складывает два числа и возвращает результат строкой."""
    result = a + b
    return f"Сумма {a} и {b} равна {result}"


# ---------------------------------------------------------------
# ИНСТРУМЕНТ 2: поиск по глоссарию
# ---------------------------------------------------------------
def lookup_glossary(term: str) -> str:
    """Ищет определение термина в глоссарии (файл glossary.json)."""
    if not os.path.exists(GLOSSARY_PATH):
        return "Глоссарий не найден."
    with open(GLOSSARY_PATH, "r", encoding="utf-8") as f:
        glossary = json.load(f)
    key = term.strip().lower()
    if key in glossary:
        return glossary[key]
    # Простой поиск по частичному совпадению
    for k, v in glossary.items():
        if key in k or k in key:
            return v
    return f"Термин «{term}» не найден в глоссарии."


# ---------------------------------------------------------------
# ОПИСАНИЕ ИНСТРУМЕНТОВ ДЛЯ МОДЕЛИ (JSON Schema)
# ---------------------------------------------------------------
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Складывает два числа. Используй, когда нужно посчитать сумму.",
            "parameters": {
                "type": "object",
                "required": ["a", "b"],
                "properties": {
                    "a": {"type": "number", "description": "Первое число"},
                    "b": {"type": "number", "description": "Второе число"},
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_glossary",
            "description": "Ищет определение психологического термина в глоссарии. Используй, когда нужно объяснить термин.",
            "parameters": {
                "type": "object",
                "required": ["term"],
                "properties": {
                    "term": {
                        "type": "string",
                        "description": "Термин, например «катастрофизация»",
                    }
                },
            },
        },
    },
]


def call_model(messages, tools=TOOLS):
    """Отправляет сообщения в Ollama и возвращает ответ модели целиком."""
    payload = {
        "model": MODEL,
        "messages": messages,
        "tools": tools,
        "stream": False,
        "options": {"temperature": 0.2},
    }
    response = requests.post(CHAT_URL, json=payload, timeout=300)
    response.raise_for_status()
    return response.json()["message"]


def agent_loop(user_question, max_steps=5):
    """Главный цикл агента: модель решает, что делать, пока не даст ответ."""
    messages = [{"role": "user", "content": user_question}]

    for step in range(1, max_steps + 1):
        print(f"\n--- Шаг {step} ---")
        message = call_model(messages)

        # Если модель НЕ просит инструмент — она даёт финальный ответ
        if not message.get("tool_calls"):
            print("Модель дала финальный ответ.")
            return message.get("content") or ""

        # Сообщение ассистента с tool_calls добавляем в историю ОДИН раз —
        # до цикла по вызовам (см. пояснение в Шаге 2.3).
        print("Модель решила вызвать инструмент(ы):")
        messages.append(message)

        # Обрабатываем каждый вызов инструмента
        for tool_call in message["tool_calls"]:
            name = tool_call["function"]["name"]
            args = tool_call["function"]["arguments"]
            print(f"  {name} с аргументами {args}")

            if name == "add_numbers":
                result = add_numbers(**args)
            elif name == "lookup_glossary":
                result = lookup_glossary(**args)
            else:
                result = f"Неизвестный инструмент: {name}"

            # Добавляем результат работы инструмента
            messages.append(
                {
                    "role": "tool",
                    "tool_name": name,
                    "content": str(result),
                }
            )

    return "Агент не смог завершить работу за отведённое число шагов."


if __name__ == "__main__":
    print("Простой агент: калькулятор + глоссарий. Введите вопрос или 'выход'.")
    while True:
        question = input("\nВы: ").strip()
        if question.lower() in ("выход", "exit", "quit"):
            break
        if not question:
            continue

        answer = agent_loop(question)
        print("\nОтвет агента:")
        print(answer)
```

**Что изменилось по сравнению с Шагом 2:**
- Добавлены `import json` и `import os`.
- Добавлена константа `GLOSSARY_PATH`.
- Добавлена функция `lookup_glossary`.
- В список `TOOLS` добавлено второе описание — для `lookup_glossary`.
- В цикле обработки добавлена ветка `elif name == "lookup_glossary"`.
- `messages.append(message)` остался вынесенным из цикла `for tool_call` (как и в Шаге 2).
- `message.get("content") or ""` — та же защита от `null`.

### 3.3. Проверьте

Запустите снова:
```powershell
python simple_agent.py
```

Спросите: **«Что такое катастрофизация?»**

Агент должен вызвать `lookup_glossary`, получить определение из файла `glossary.json` и ответить на его основе.

Теперь спросите что-то вроде **«Сколько будет 5 плюс 7 и что такое сверхобобщение?»** — модель может вызвать **два инструмента за один шаг**. Это и есть тот случай, ради которого мы вынесли `messages.append(message)` из цикла: история останется корректной.

**Если `lookup_glossary` возвращает «Глоссарий не найден»** — проверьте, что `glossary.json` лежит в той же папке, откуда вы запускаете скрипт (то есть в `code/`), и что вы запускаете скрипт из этой папки.

---

## Шаг 4. Мультиагентная проверка: ответчик → критик → корректор

Теперь главное. Мы построим цепочку из трёх **ролей**, которые последовательно обрабатывают один и тот же вопрос. Каждая роль — это отдельный вызов модели с **разным системным промптом**. Это простейшая мультиагентная схема без внешних фреймворков.

**Зачем это нужно?** Даже с инструментами модель может ошибиться: перепутать термин, дать совет, который не следует из глоссария, или ответить слишком категорично. Критик смотрит на ответ «свежим взглядом» и ищет проблемы. Корректор исправляет их, опираясь на замечания.

### 4.1. Создайте файл `multi_agent.py` в папке `code`

В левой панели Explorer наведите курсор на папку `code` → нажмите иконку **New File** → введите `multi_agent.py` → `Enter`.

### 4.2. Вставьте полный код (копируйте целиком, ничего не пропуская)

```python
import json
import os
import requests

# ---------------------------------------------------------------
# НАСТРОЙКИ
# ---------------------------------------------------------------
CHAT_URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5:3b"          # модель, поддерживающая tool calling
GLOSSARY_PATH = "glossary.json"


# ---------------------------------------------------------------
# ИНСТРУМЕНТ 1: калькулятор
# ---------------------------------------------------------------
def add_numbers(a: float, b: float) -> str:
    """Складывает два числа и возвращает результат строкой."""
    result = a + b
    return f"Сумма {a} и {b} равна {result}"


# ---------------------------------------------------------------
# ИНСТРУМЕНТ 2: поиск по глоссарию
# ---------------------------------------------------------------
def lookup_glossary(term: str) -> str:
    """Ищет определение термина в глоссарии (файл glossary.json)."""
    if not os.path.exists(GLOSSARY_PATH):
        return "Глоссарий не найден."
    with open(GLOSSARY_PATH, "r", encoding="utf-8") as f:
        glossary = json.load(f)
    key = term.strip().lower()
    if key in glossary:
        return glossary[key]
    # Простой поиск по частичному совпадению
    for k, v in glossary.items():
        if key in k or k in key:
            return v
    return f"Термин «{term}» не найден в глоссарии."


# ---------------------------------------------------------------
# ОПИСАНИЕ ИНСТРУМЕНТОВ ДЛЯ МОДЕЛИ (JSON Schema)
# ---------------------------------------------------------------
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "add_numbers",
            "description": "Складывает два числа. Используй, когда нужно посчитать сумму.",
            "parameters": {
                "type": "object",
                "required": ["a", "b"],
                "properties": {
                    "a": {"type": "number", "description": "Первое число"},
                    "b": {"type": "number", "description": "Второе число"},
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_glossary",
            "description": "Ищет определение психологического термина в глоссарии. Используй, когда нужно объяснить термин.",
            "parameters": {
                "type": "object",
                "required": ["term"],
                "properties": {
                    "term": {
                        "type": "string",
                        "description": "Термин, например «катастрофизация»",
                    }
                },
            },
        },
    },
]


# ---------------------------------------------------------------
# ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ: один вызов модели с system-промптом
# ---------------------------------------------------------------
def call_model(system_text, user_text, tools=None):
    """Отправляет в Ollama system + user сообщения и возвращает message.

    Параметр tools оставлен для будущих улучшений: если, например,
    критик захочет сам сверяться с глоссарием через инструмент.
    В текущем варианте критик и корректор работают без инструментов.
    """
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "options": {"temperature": 0.2},
    }
    if tools:
        payload["tools"] = tools
    response = requests.post(CHAT_URL, json=payload, timeout=300)
    response.raise_for_status()
    return response.json()["message"]


# ---------------------------------------------------------------
# АГЕНТ 1: ОТВЕТЧИК — может вызывать инструменты
# ---------------------------------------------------------------
def answer_agent(question):
    """Отвечает на вопрос, используя инструменты при необходимости."""
    system_text = (
        "Ты — помощник по психологии. Отвечай кратко и по делу. "
        "Если нужно посчитать — используй калькулятор add_numbers. "
        "Если нужно объяснить термин — используй глоссарий lookup_glossary."
    )
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": question},
    ]

    for _ in range(5):
        payload = {
            "model": MODEL,
            "messages": messages,
            "tools": TOOLS,
            "stream": False,
            "options": {"temperature": 0.2},
        }
        response = requests.post(CHAT_URL, json=payload, timeout=300)
        response.raise_for_status()
        message = response.json()["message"]

        # Если модель не просит инструмент — это финальный ответ
        if not message.get("tool_calls"):
            return message.get("content") or ""

        # Сообщение ассистента с tool_calls добавляем ОДИН раз —
        # до цикла по вызовам. Иначе при двух вызовах за шаг
        # история сломается.
        messages.append(message)

        # Обрабатываем каждый вызов инструмента
        for tool_call in message["tool_calls"]:
            name = tool_call["function"]["name"]
            args = tool_call["function"]["arguments"]
            if name == "add_numbers":
                result = add_numbers(**args)
            elif name == "lookup_glossary":
                result = lookup_glossary(**args)
            else:
                result = f"Неизвестный инструмент: {name}"

            # Добавляем результат работы инструмента
            messages.append(
                {"role": "tool", "tool_name": name, "content": str(result)}
            )

    return "Ответчик не смог завершить работу за отведённое число шагов."


# ---------------------------------------------------------------
# АГЕНТ 2: КРИТИК — проверяет ответ, но НЕ переписывает его
# ---------------------------------------------------------------
def critic_agent(question, answer):
    """Ищет проблемы в ответе: ошибки, выдумки, слишком категоричные советы."""
    system_text = (
        "Ты — строгий, но справедливый критик. Твоя задача — проверить ответ помощника. "
        "Найди: (1) фактические ошибки; (2) утверждения, которых нет в глоссарии; "
        "(3) слишком категоричные советы. "
        "Не переписывай ответ. Просто перечисли проблемы списком. "
        "Если проблем нет — напиши «Проблем не найдено»."
    )
    user_text = f"Вопрос: {question}\n\nОтвет помощника:\n{answer}"
    message = call_model(system_text, user_text)
    return message.get("content") or ""


# ---------------------------------------------------------------
# АГЕНТ 3: КОРРЕКТОР — исправляет ответ с учётом замечаний
# ---------------------------------------------------------------
def corrector_agent(question, answer, critique):
    """Переписывает ответ, устраняя проблемы, найденные критиком."""
    system_text = (
        "Ты — редактор. Перепиши ответ помощника, устранив проблемы из списка критика. "
        "Не добавляй новых фактов, которых не было в исходном ответе или в глоссарии. "
        "Если критик сказал «Проблем не найдено», верни исходный ответ без изменений."
    )
    user_text = (
        f"Вопрос: {question}\n\n"
        f"Исходный ответ:\n{answer}\n\n"
        f"Замечания критика:\n{critique}"
    )
    message = call_model(system_text, user_text)
    return message.get("content") or ""


# ---------------------------------------------------------------
# ГЛАВНЫЙ ЦИКЛ
# ---------------------------------------------------------------
def main():
    print("Мультиагентная система: ответчик → критик → корректор.")
    print("Введите вопрос или 'выход'.\n")

    while True:
        question = input("Вы: ").strip()
        if question.lower() in ("выход", "exit", "quit"):
            print("До свидания!")
            break
        if not question:
            continue

        try:
            print("\n[1/3] Ответчик готовит ответ...")
            answer = answer_agent(question)
            print("Ответчик:", answer)

            # Если ответчик не справился — критика и корректора
            # запускать бессмысленно: они будут анализировать
            # сообщение об ошибке, а не реальный ответ.
            if answer.startswith("Ответчик не смог"):
                print("Пропускаю критика и корректора: ответчик не справился.")
                continue

            print("\n[2/3] Критик проверяет...")
            critique = critic_agent(question, answer)
            print("Критик:", critique)

            print("\n[3/3] Корректор исправляет...")
            final = corrector_agent(question, answer, critique)
            print("\n=== Итоговый ответ ===")
            print(final)
            print()

        except requests.exceptions.ConnectionError:
            print("Ошибка: Ollama не запущена. Проверьте, что программа Ollama работает.")
        except requests.exceptions.Timeout:
            print("Модель не ответила за отведённое время. Попробуйте ещё раз.")
        except requests.exceptions.HTTPError as e:
            print(f"HTTP-ошибка: {e}. Проверьте, что модель '{MODEL}' скачана (ollama list).")


if __name__ == "__main__":
    main()
```

### 4.3. Убедитесь, что рядом лежит `glossary.json`

`multi_agent.py` использует глоссарий из Шага 3. Если вы ещё не создали файл `glossary.json` в папке `code`, создайте его сейчас и вставьте:

```json
{
  "катастрофизация": "Когнитивное искажение: представление о том, что произойдёт самое худшее из возможного, без учёта других исходов.",
  "сверхобобщение": "Когнитивное искажение: вывод о том, что одно негативное событие будет повторяться всегда.",
  "персонализация": "Когнитивное искажение: принятие на себя ответственности за события, которые от вас не зависят.",
  "чтение мыслей": "Когнитивное искажение: уверенность в том, что вы знаете, о чём думают другие, без достаточных оснований."
}
```

В папке `code` теперь должно быть **три файла**:
```
code/
├── glossary.json
├── simple_agent.py
└── multi_agent.py
```

### 4.4. Запустите и проверьте

1. Убедитесь, что Ollama работает (значок в трее).
2. В терминале VS Code перейдите в папку `code` (если ещё не там):
   ```powershell
   cd code
   ```
3. Запустите:
   ```powershell
   python multi_agent.py
   ```

**Тестовые вопросы (по одному):**

- `Что такое сверхобобщение?` — ответчик вызовет `lookup_glossary`, критик сверит с глоссарием, корректор при необходимости поправит.
- `Сколько будет 89 плюс 34?` — ответчик вызовет `add_numbers`, критик проверит арифметику.
- `Расскажи про катастрофизацию и приведи пример` — здесь интересно: если ответчик добавит пример «от себя», критик может отметить, что примера нет в глоссарии.

**Что вы увидите в терминале** (примерный вид):
```
Вы: Что такое сверхобобщение?

[1/3] Ответчик готовит ответ...
Ответчик: Сверхобобщение — это когнитивное искажение, при котором человек делает вывод, что одно негативное событие будет повторяться всегда.

[2/3] Критик проверяет...
Критик: Проблем не найдено. Определение соответствует глоссарию.

[3/3] Корректор исправляет...

=== Итоговый ответ ===
Сверхобобщение — это когнитивное искажение, при котором человек делает вывод, что одно негативное событие будет повторяться всегда.
```

Если увидели три блока и итоговый ответ — цепочка работает.

**Если появилась ошибка:**
- `ConnectionError` — Ollama не запущена. Откройте её.
- `HTTP-ошибка: 404` — модель `qwen2.5:3b` не скачана. Выполните `ollama pull qwen2.5:3b`.
- `lookup_glossary` возвращает «Глоссарий не найден» — проверьте, что `glossary.json` лежит **в той же папке**, откуда вы запускаете скрипт (то есть в `code/`).

---

## Шаг 5. Что сдавать

1. Папка `agent_psychology` с файлами: `code/simple_agent.py`, `code/multi_agent.py`, `code/glossary.json`.
2. **`README.md`** (полстраницы):
   - какие инструменты реализованы;
   - как запустить каждого агента;
   - 3 примера работы мультиагентной цепочки (можно скриншотами терминала).
3. **Короткий отчёт** (1 страница):
   - в каких случаях критик находил проблемы, а в каких — нет;
   - где цепочка работала хорошо, а где — плохо;
   - что бы вы изменили в системных промптах.

---

## Шаг 6. Критерии оценки

| Критерий | Вес |
|---|---|
| Простой агент вызывает инструменты и завершает цикл | 30% |
| Мультиагентная цепочка работает end-to-end | 30% |
| Критик находит осмысленные проблемы (не просто «всё хорошо») | 20% |
| Понятность кода и README | 10% |
| Рефлексия в отчёте | 10% |

---

## Приложение. Краткая справка по актуальному API tool calling (на 14.11.2026)

- **Эндпоинт:** `POST http://localhost:11434/api/chat`. Поле `tools` — массив описаний функций в формате JSON Schema (OpenAI-совместимый).
- **Ответ:** если модель хочет вызвать инструмент, в `message.tool_calls` будет массив вызовов. Каждый вызов содержит `function.name` и `function.arguments`. Аргументы приходят как объект (словарь), и это важно учитывать при передаче в функцию через `**args`.
- **Обратный вызов:** результат инструмента добавляется в `messages` с ролью `tool` и полем `tool_name`.
- **Сообщение ассистента с `tool_calls`** добавляется в историю **один раз до цикла** по вызовам. Это критично: если модель вернёт два вызова за шаг, дублирование сломает диалог.
- **Цикл:** повторяйте вызов, пока в ответе есть `tool_calls`. Ограничивайте число шагов (например, 5), чтобы избежать зацикливания.
- **`content: null`:** при вызове инструментов `message.content` может быть `null`, а не пустой строкой. Поэтому в коде используется `message.get("content") or ""`, а не `message.get("content", "")`.
- **Модели:** `qwen2.5` (3b, 7b), `llama3.1` (8b+), `mistral` (7b), `qwen3`, `gemma4` и другие — поддерживают tool calling. `phi3:mini` — не лучший выбор для этого проекта.
- **Python SDK:** `pip install ollama -U`. Позволяет передавать функции напрямую в `tools=[...]`, а SDK сам строит схему из аннотаций типов и docstring.

---

## Возможные проблемы и что с ними делать

| Симптом | Причина | Что делать |
|---|---|---|
| Модель отвечает «172», но не вызывает инструмент | Модель «угадала» ответ или не поддерживает tool calling | Используйте `qwen2.5:3b` или другую модель из списка; задайте вопрос с большими числами |
| `HTTPError: 404` | Модель не скачана | `ollama pull qwen2.5:3b` |
| `ConnectionError` | Ollama не запущена | Откройте Ollama (значок в трее) |
| `lookup_glossary` возвращает «Глоссарий не найден» | Запуск не из папки `code/` или файл не создан | Перейдите в `code/` (`cd code`) и проверьте наличие `glossary.json` |
| Критик всегда говорит «Проблем не найдено» | Слишком мягкий системный промпт или слабая модель | Усильте формулировку («Ты — очень строгий критик»), попробуйте модель побольше (`qwen2.5:7b`) |
| Модель «зациклилась» и вызывает инструмент снова и снова | Ограничение `max_steps=5` сработало | Это нормальная защита; попробуйте переформулировать вопрос |
| Долгий ответ (больше минуты) | Маленькая модель на слабом ноутбуке | Уменьшите размер модели или закройте лишние программы |



## Промпт 1. Добавить третий инструмент — чтение файла с заметками

**Зачем:** сейчас агент умеет считать и искать в глоссарии. Инструмент `read_notes(topic)` научит его работать с файловой системой — это следующий естественный шаг после двух «игрушечных» инструментов.

**Готовый промпт:**

```text
Добавь в мой simple_agent.py третий инструмент read_notes(topic), который читает текст из файла notes/<topic>.txt и возвращает его содержимое.

Требования:
1. Если файла нет — верни сообщение: «Заметок по теме <topic> не найдено».
2. Опиши инструмент в списке TOOLS в формате JSON Schema, как остальные два.
3. В цикле обработки tool_calls добавь ветку elif name == "read_notes".
4. Не забудь добавить import os (если его ещё нет).
5. Покажи полный обновлённый simple_agent.py с комментариями на русском.

Вот мой текущий simple_agent.py:
[вставьте полный код simple_agent.py]
```

**Что этот промпт тренирует:** понимание того, что «инструмент» — это обычная Python-функция, а добавление нового инструмента — это три шага: функция, описание в TOOLS, ветка в цикле обработки.


## Промпт 2. Ограничить длину результата инструмента

**Зачем:** если `read_notes` вернёт длинный текст, он «забьёт» контекст модели. Практика 2026 года — обрезать результаты инструментов, оставляя начало и конец, и добавлять маркер `<<<TRUNCATED>>>`, чтобы модель понимала, что часть текста опущена.

**Готовый промпт:**

```text
Добавь в simple_agent.py функцию truncate_tool_result(text, max_len=4000), которая обрезает текст до max_len символов.

Требования:
1. Оставляй первые 1500 символов и последние 1500 символов.
2. В середину вставляй маркер: "\n\n<<<TRUNCATED>>> Остальная часть результата опущена для экономии контекста.\n\n".
3. Применяй эту функцию ко всем результатам инструментов ПЕРЕД добавлением в messages (и для add_numbers, и для lookup_glossary, и для read_notes).
4. Покажи полный обновлённый simple_agent.py с комментариями на русском.

Вот мой текущий simple_agent.py:
[вставьте полный код simple_agent.py]
```

**Что этот промпт тренирует:** понимание того, что контекст модели — ограниченный ресурс, и что результаты инструментов нужно «упаковывать» аккуратно.


## Промпт 3. Логирование шагов агента в файл

**Зачем:** когда цепочка из трёх агентов работает «вслепую», сложно понять, где именно произошла ошибка. Лог в файл `agent.log` покажет, какой инструмент вызывался, с какими аргументами и что вернул.

**Готовый промпт:**

```text
Добавь в multi_agent.py логирование в файл agent.log.

Требования:
1. Используй стандартный модуль logging (никаких внешних библиотек).
2. Для каждого шага записывай: время, роль агента (ответчик / критик / корректор), текст запроса к модели, ответ модели, какие инструменты вызывались (если были).
3. Формат строки: "2026-11-14 15:30:22 | ответчик | инструмент: lookup_glossary | аргументы: {'term': 'катастрофизация'}".
4. Логирование не должно ломать работу программы: если файл не удаётся открыть, просто выведи предупреждение в консоль.
5. Покажи полный обновлённый multi_agent.py с комментариями на русском.

Вот мой текущий multi_agent.py:
[вставьте полный код multi_agent.py]
```

**Что этот промпт тренирует:** понимание того, что наблюдаемость (observability) — обязательная часть агентных систем, а не «дополнительная опция».


## Промпт 4. Сделать критика «строже» с помощью глоссария

**Зачем:** сейчас критик не имеет доступа к глоссарию и проверяет ответ «на глаз». Если дать ему инструмент `lookup_glossary`, он сможет **сверить** определение термина с эталоном, а не просто догадываться. Это качественно меняет его работу.

**Готовый промпт:**

```text
Добавь в critic_agent из multi_agent.py доступ к инструменту lookup_glossary.

Требования:
1. Критик должен при проверке терминов вызывать lookup_glossary и сравнивать определение в ответе с определением в глоссарии.
2. Если они расходятся — указывать это в замечаниях: «Определение термина X расходится с глоссарием: в ответе ..., в глоссарии ...».
3. Используй ту же логику tool calling, что в answer_agent (цикл по tool_calls, добавление message и результата инструмента в историю).
4. Важно: сообщение ассистента с tool_calls добавляется в историю ОДИН раз до цикла по вызовам (как в answer_agent).
5. Покажи полный обновлённый multi_agent.py (или хотя бы обновлённые critic_agent и вспомогательную функцию call_model с поддержкой tools) с комментариями на русском.

Вот мой текущий multi_agent.py:
[вставьте полный код multi_agent.py]
```

**Что этот промпт тренирует:** понимание того, что разные роли могут иметь разные инструменты. Критик с доступом к эталону — это уже не «мнение», а проверка.


## Промпт 5. Автоматическая оценка: насколько корректор улучшил ответ

**Зачем:** чтобы понять, работает ли цепочка, нужно сравнить ответчик и корректор. Простейший способ — попросить **отдельного агента-судью** оценить оба ответа по шкале 1–10. Это учит мыслить метриками, а не «на глаз».

**Готовый промпт:**

```text
Добавь в multi_agent.py функцию judge_agent(question, answer, final_answer), которая просит модель оценить, насколько итоговый ответ лучше исходного.

Требования:
1. Системный промпт судьи: «Ты — беспристрастный судья. Сравни два ответа на один вопрос. Оцени по шкале от 1 до 10, насколько второй ответ лучше первого. Если второй ответ не лучше, поставь 1–5. Если лучше — 6–10. Обоснуй оценку одним предложением».
2. В main() после получения final вызови judge_agent и выведи оценку и обоснование.
3. Если ответчик не справился (answer.startswith("Ответчик не смог")), судью не вызывай.
4. Покажи полный обновлённый multi_agent.py с комментариями на русском.

Вот мой текущий multi_agent.py:
[вставьте полный код multi_agent.py]
```

**Что этот промпт тренирует:** понимание того, что «улучшение» — это измеримая величина, а не субъективное ощущение.


## Промпт 6. Перевести агента на официальный Python SDK Ollama

**Зачем:** `requests` — это «ручная» работа: вы сами строите JSON Schema, сами разбираете `tool_calls`. Официальный SDK `ollama` (`pip install ollama -U`) позволяет передавать функции напрямую в `tools=[...]`, а схему он строит сам из аннотаций типов и docstring. Код становится короче и надёжнее.

**Готовый промпт:**

```text
Перепиши simple_agent.py с использованием официального Python SDK ollama.

Требования:
1. Установка: pip install ollama -U (покажи команду).
2. Используй from ollama import chat.
3. Опиши инструменты как обычные Python-функции с аннотациями типов и docstring — SDK сам построит схему.
4. Передавай функции напрямую в tools=[add_numbers, lookup_glossary].
5. Сохрани всю текущую логику: цикл по tool_calls, добавление message и результата инструмента в историю, ограничение max_steps.
6. Важно: сообщение ассистента с tool_calls добавляется в историю ОДИН раз до цикла по вызовам.
7. Покажи полный обновлённый simple_agent.py и кратко объясни, что изменилось по сравнению с requests.

Вот мой текущий simple_agent.py на requests:
[вставьте полный код simple_agent.py]
```

**Что этот промпт тренирует:** понимание того, что SDK — это не «магия», а удобная обёртка, которая убирает рутину и снижает вероятность ошибок в JSON Schema.


## Промпт 7. Добавить валидацию аргументов инструментов

**Зачем:** маленькие локальные модели иногда присылают аргументы неверного типа (например, число строкой `"30"` вместо `30`) или выдумывают несуществующее имя инструмента. Валидация перед вызовом функции делает агента устойчивее и даёт понятную ошибку вместо `TypeError`.

**Готовый промпт:**

```text
Добавь в simple_agent.py валидацию аргументов перед вызовом инструмента.

Требования:
1. Создай функцию validate_tool_args(name, args), которая проверяет:
   - для add_numbers: args содержит ключи "a" и "b", и оба значения — числа (int или float). Если пришла строка с числом — преобразуй в число. Если не удалось — верни ошибку.
   - для lookup_glossary: args содержит ключ "term", значение — непустая строка.
2. Если валидация не прошла, верни в messages сообщение с ролью "tool" и текстом: «Ошибка аргументов: <описание>. Попробуй вызвать инструмент снова с корректными аргументами».
3. Не вызывай инструмент, если валидация не прошла.
4. Покажи полный обновлённый simple_agent.py с комментариями на русском.

Вот мой текущий simple_agent.py:
[вставьте полный код simple_agent.py]
```

**Что этот промпт тренирует:** понимание того, что модель может ошибаться в формате, и что код должен защищаться от этого, а не падать с треском.


### Как пользоваться этими промптами (памятка)

1. **Выберите один промпт** и скопируйте его в DeepSeek или ChatGPT.
2. **Приложите свой текущий код** (`simple_agent.py` или `multi_agent.py`) в конец промпта.
3. Получите обновлённый код, **сравните** с исходным и прочитайте комментарии.
4. **Запустите** и проверьте, что улучшение действительно работает.
5. **Сохраните копию** перед изменениями: `simple_agent_v1.py`, `multi_agent_v2.py` — чтобы можно было откатиться.

Начинайте с **Промпта 1** (третий инструмент) и **Промпта 2** (обрезка результата) — они дадут наибольший практический эффект при минимальных изменениях. Промпт 4 (критик с глоссарием) — самый интересный с точки зрения «агентности»: он показывает, как роль меняет поведение системы.